# OpenCLIP model binary classificaiton on reduced dataset

## Imports

In [1]:
import torch
import os

from PIL import Image
import open_clip
import numpy as np
import pandas as pd

import joblib

from nazi_symbols_classification.training.data_preparation import get_image_paths
from nazi_symbols_classification.training.evaluation import get_top1_evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Data Preparation

In [2]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-detection-simple", ("train", "test", "val"))

In [3]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/val')]

In [4]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

## Load model

In [5]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.to("cuda")

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

## Define classify_image function

In [9]:
def classify_image(image_path, prompts):
    text = tokenizer(prompts)
    with torch.no_grad(), torch.autocast("cuda"):
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image.cuda())
        text_features = model.encode_text(text.cuda())  # type: ignore
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        return {k:round(v.item(), 3) for k, v in dict(zip(prompts, text_probs[0])).items()}

In [7]:
prompts = {
    "A single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "nazi",
    "A skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "nazi",
    "A black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "nazi",
    "An image containing no Nazi-related content.": "non-nazi"
}

In [10]:
tmp_result = {"nazi": 0.0, "non-nazi": 0.0}
classify_result = classify_image(test_images[0], prompts.keys())
for k, v in classify_result.items():
    tmp_result[prompts[k]] += v
sorted(tmp_result.items(), key=lambda x: x[1], reverse=True)[0][0]

'nazi'

## Classify all the images from test images for evaluation purpose

In [11]:
%%time

probs = []

for image_path in test_images:
    tmp_result = {"nazi": 0.0, "non-nazi": 0.0}
    classify_result = classify_image(image_path, prompts.keys())
    for k, v in classify_result.items():
        tmp_result[prompts[k]] += v
    probs.append(tmp_result["nazi"])

CPU times: user 56min 18s, sys: 4.34 s, total: 56min 22s
Wall time: 4min 3s


Store the outputs

In [12]:
result = dict(y_true=[int(label == "nazi-symbol") for label in y_test], 
              probs=probs, 
              outputs=[int(prob > 0.5) for prob in probs])

joblib.dump(result, "openclip-output/open_clip_result_binary_reduced.joblib")

['openclip-output/open_clip_result_v2-binary-simple.joblib']

## Print classification reports and calculate metrics

In [15]:
print(classification_report([int(label == "nazi-symbol") for label in y_test], result["outputs"], digits=3))

              precision    recall  f1-score   support

           0      0.992     0.870     0.927     14813
           1      0.025     0.338     0.047       148

    accuracy                          0.865     14961
   macro avg      0.509     0.604     0.487     14961
weighted avg      0.983     0.865     0.919     14961

